<a href="https://colab.research.google.com/github/MR-just01/Llama3.2-Reasoning/blob/main/notebooks/03_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Merging

This notebook combines all standardized reasoning datasets into a single dataset for QLoRA fine-tuning.

Datasets:
- GSM8K
- StrategyQA
- ARC Challenge
- AQUA-RAT

Pipeline:
1. Load standardized datasets
2. Verify schema
3. Balance datasets
4. Merge datasets
5. Shuffle merged dataset
6. Validate merged dataset
7. Save final training dataset

In [1]:
import pandas as pd
import numpy as np

In [5]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning.git

Cloning into 'Llama3.2-Reasoning'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 69 (delta 34), reused 33 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 20.54 MiB | 14.86 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [6]:
%cd /content/Llama3.2-Reasoning

/content/Llama3.2-Reasoning


In [7]:
!find data

data
data/raw
data/raw/ReadME.md
data/splits
data/splits/ReadME.md
data/processed
data/processed/startegyQA_standardized.csv
data/processed/gsm8k_standardized.csv
data/processed/aqua_rat_standardized.csv
data/processed/reasoning_dataset.csv
data/processed/arc_challenge_standardized.csv


In [8]:
import pandas as pd

gsm = pd.read_csv("data/processed/gsm8k_standardized.csv",
  keep_default_na=False)

strategy = pd.read_csv(
    "data/processed/startegyQA_standardized.csv",
    keep_default_na=False
)

arc = pd.read_csv(
    "data/processed/arc_challenge_standardized.csv",
    keep_default_na=False
)

aqua = pd.read_csv(
    "data/processed/aqua_rat_standardized.csv",
    keep_default_na=False
)


print(aqua.isnull().sum())

instruction    0
input          0
reasoning      0
answer         0
dataset        0
task_type      0
dtype: int64


In [9]:
print("GSM8K:", gsm.shape)
print("StrategyQA:", strategy.shape)
print("ARC:", arc.shape)
print("AQUA-RAT:", aqua.shape)

GSM8K: (7473, 6)
StrategyQA: (1603, 6)
ARC: (1119, 6)
AQUA-RAT: (97467, 6)


In [10]:
datasets = {
    "GSM8K": gsm,
    "StrategyQA": strategy,
    "ARC": arc,
    "AQUA-RAT": aqua
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())


GSM8K
['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']

StrategyQA
['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']

ARC
['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']

AQUA-RAT
['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']


In [11]:
stats = pd.DataFrame({
    "Dataset": [
        "GSM8K",
        "StrategyQA",
        "ARC Challenge",
        "AQUA-RAT"
    ],
    "Rows": [
        len(gsm),
        len(strategy),
        len(arc),
        len(aqua)
    ]
})

stats

,Dataset,Rows
0,GSM8K,7473
1,StrategyQA,1603
2,ARC Challenge,1119
3,AQUA-RAT,97467


## Dataset Balancing

The four reasoning datasets have significantly different sizes.

AQUA-RAT contains substantially more examples than the other datasets and would dominate the training process if merged without modification.

To create a more balanced training corpus, all samples from the smaller datasets are retained, while a random subset of AQUA-RAT is selected.

A fixed random seed is used to ensure reproducibility.

In [12]:
RANDOM_SEED = 42

aqua_sampled = aqua.sample(
    n=20000,
    random_state=RANDOM_SEED
)

print(aqua_sampled.shape)

(20000, 6)


In [13]:
balanced_stats = pd.DataFrame({
    "Dataset": [
        "GSM8K",
        "StrategyQA",
        "ARC Challenge",
        "AQUA-RAT (Sampled)"
    ],
    "Rows": [
        len(gsm),
        len(strategy),
        len(arc),
        len(aqua_sampled)
    ]
})

balanced_stats

,Dataset,Rows
0,GSM8K,7473
1,StrategyQA,1603
2,ARC Challenge,1119
3,AQUA-RAT (Sampled),20000


In [14]:
merged_df = pd.concat(
    [
        gsm,
        strategy,
        arc,
        aqua_sampled
    ],
    ignore_index=True
)

The merged training dataset contains **30,195** examples after balancing.

Dataset distribution:

- GSM8K: 7,473
- StrategyQA: 1,603
- ARC Challenge: 1,119
- AQUA-RAT (Sampled): 20,000

Total: 30,195 examples.

In [15]:
merged_df.shape

(30195, 6)

In [16]:
merged_df = merged_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [17]:
merged_df.head(40)

,instruction,input,reasoning,answer,dataset,task_type
0,Solve the following math reasoning problem ste...,Nicole collected 400 Pokemon cards. Cindy coll...,150,150,gsm8k,math_reasoning
1,Solve the following math reasoning problem ste...,James had two browsers on his computer. In eac...,60,60,gsm8k,math_reasoning
2,Solve the following multiple-choice math reaso...,The 100-milliliter solution of sugar and water...,In the original solution the amount of sugar i...,50,AQUA-RAT,math_reasoning
3,Solve the following multiple-choice math reaso...,"If 0.75 : x :: 5 : 8, then x is equal to:\n\nC...",Explanation:\n(x x 5) = (0.75 x 8)\nx=6/5\n=1....,1.2,AQUA-RAT,math_reasoning
4,Solve the following multiple-choice math reaso...,The price of Darjeeling tea (in rupees per kil...,Explanation :\nPrice of Darjeeling tea (in rup...,May 20,AQUA-RAT,math_reasoning
5,Solve the following math reasoning problem ste...,"If it takes 20 minutes to paint a house, how m...",9,9,gsm8k,math_reasoning
6,Solve the following multiple-choice math reaso...,A person walking takes 26 steps to come down o...,26+30n=18+34n\n12n=8\nn=2/3 substitute n valve...,46 steps,AQUA-RAT,math_reasoning
7,Solve the following math reasoning problem ste...,Lucas wants to get a dog but his parents think...,10,10,gsm8k,math_reasoning
8,Solve the following math reasoning problem ste...,Wendy's truck has a gas tank that can hold 20 ...,18,18,gsm8k,math_reasoning
9,Solve the following multiple-choice math reaso...,The least whole number which when subtracted f...,"Explanation:\nLet x is subtracted.\nThen,\n(6−...",3,AQUA-RAT,math_reasoning


In [18]:
merged_df["dataset"].value_counts()

,count
dataset,
AQUA-RAT,20000
gsm8k,7473
StrategyQA,1603
ARC-Challenge,1119


Validate the merged datasets **bold text**

In [19]:
# print(merged_df.shape)

merged_df.isnull().sum()
# merged_df.dtypes

,0
instruction,0
input,0
reasoning,0
answer,0
dataset,0
task_type,0


In [20]:
duplicate_inputs = merged_df[
    merged_df["input"].duplicated(keep=False)
]

print("Duplicate inputs:", len(duplicate_inputs))

Duplicate inputs: 4


In [21]:
duplicate_inputs[
    ["dataset", "input", "answer", "reasoning"]
].sort_values("input")

,dataset,input,answer,reasoning
6713,ARC-Challenge,How many times does Earth rotate on its axis i...,once,Not provided
18254,ARC-Challenge,How many times does Earth rotate on its axis i...,once,Not provided
3853,AQUA-RAT,Mr. Jones sold two pipes at $1.20 each. Based ...,broke even,Solution:\n20 % profit on $ 1.20\n= $ 20/100 ×...
8656,AQUA-RAT,Mr. Jones sold two pipes at $1.20 each. Based ...,broke even,20 % profit on $ 1.20\n= $ 20/100 × 1.20\n= $ ...


In [22]:
merged_df = merged_df.drop_duplicates(
    subset=["dataset", "input"],
    keep="first"
).reset_index(drop=True)

In [23]:
subset=["dataset", "input"]

In [24]:
duplicate_inputs = merged_df[
    merged_df.duplicated(
        subset=["dataset", "input"],
        keep=False
    )
]

print(len(duplicate_inputs))

0


In [25]:
merged_df.info()
merged_df.isnull().sum()
merged_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30193 entries, 0 to 30192
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  30193 non-null  object
 1   input        30193 non-null  object
 2   reasoning    30193 non-null  object
 3   answer       30193 non-null  object
 4   dataset      30193 non-null  object
 5   task_type    30193 non-null  object
dtypes: object(6)
memory usage: 1.4+ MB


np.int64(0)

In [26]:
!ls data/processed

aqua_rat_standardized.csv	reasoning_dataset.csv
arc_challenge_standardized.csv	startegyQA_standardized.csv
gsm8k_standardized.csv


In [28]:
merged_df.to_csv("reasoning_dataset.csv", index=False)

In [29]:
from google.colab import files
files.download("reasoning_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
!ls data/processed

aqua_rat_standardized.csv	reasoning_dataset.csv
arc_challenge_standardized.csv	startegyQA_standardized.csv
gsm8k_standardized.csv
